# Concepts
This guide focuses on retrieval of text data. We will cover the following concepts:

- Documents and document loaders;
- Text splitters;
- Embeddings and Vectostore;
- retrievers.
- Generation

## Intallation

In [3]:
# pip install langchain-community pypdf

In [1]:
from langchain_community.document_loaders import PyPDFLoader #loading document
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
import ollama

## Documents and Document Loaders
- The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger documen

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Nike.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

106


In [4]:
docs[1]

Document(metadata={'source': 'Nike.pdf', 'page': 1, 'page_label': '2'}, page_content="UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K \n(Mark One)\n☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE FISCAL YEAR ENDED MAY 31, 2023 \nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE TRANSITION PERIOD FROM TO .\nCommission File No. 1-10635 \nNIKE, Inc. \n(Exact name of Registrant as specified in its charter)\nOregon 93-0584541\n(State or other jurisdiction of incorporation) (IRS Employer Identification No.)\nOne Bowerman Drive, Beaverton, Oregon 97005-6453 \n(Address of principal executive offices and zip code)\n(503) 671-6453 \n(Registrant's telephone number, including area code)\nSECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:\nClass B Common Stock NKE New York Stock Exchange\n(Title of each class) (Trading symbol) (Name of each exchange on 

In [5]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

FORM 10-KFORM 10-K

{'source': 'Nike.pdf', 'page': 0, 'page_label': '1'}


In [6]:
docs[1]

Document(metadata={'source': 'Nike.pdf', 'page': 1, 'page_label': '2'}, page_content="UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K \n(Mark One)\n☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE FISCAL YEAR ENDED MAY 31, 2023 \nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE TRANSITION PERIOD FROM TO .\nCommission File No. 1-10635 \nNIKE, Inc. \n(Exact name of Registrant as specified in its charter)\nOregon 93-0584541\n(State or other jurisdiction of incorporation) (IRS Employer Identification No.)\nOne Bowerman Drive, Beaverton, Oregon 97005-6453 \n(Address of principal executive offices and zip code)\n(503) 671-6453 \n(Registrant's telephone number, including area code)\nSECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:\nClass B Common Stock NKE New York Stock Exchange\n(Title of each class) (Trading symbol) (Name of each exchange on 

# Splitting

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

501

In [14]:
all_splits[0]

Document(metadata={'source': 'Nike.pdf', 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='FORM 10-KFORM 10-K')

In [16]:
all_splits[1]

Document(metadata={'source': 'Nike.pdf', 'page': 1, 'page_label': '2', 'start_index': 0}, page_content="UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K \n(Mark One)\n☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE FISCAL YEAR ENDED MAY 31, 2023 \nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE TRANSITION PERIOD FROM TO .\nCommission File No. 1-10635 \nNIKE, Inc. \n(Exact name of Registrant as specified in its charter)\nOregon 93-0584541\n(State or other jurisdiction of incorporation) (IRS Employer Identification No.)\nOne Bowerman Drive, Beaverton, Oregon 97005-6453 \n(Address of principal executive offices and zip code)\n(503) 671-6453 \n(Registrant's telephone number, including area code)\nSECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:\nClass B Common Stock NKE New York Stock Exchange\n(Title of each class) (Trading symbol) (Name of

In [18]:
all_splits[2]

Document(metadata={'source': 'Nike.pdf', 'page': 1, 'page_label': '2', 'start_index': 779}, page_content='Class B Common Stock NKE New York Stock Exchange\n(Title of each class) (Trading symbol) (Name of each exchange on which registered)\nSECURITIES REGISTERED PURSUANT TO SECTION 12(G) OF THE ACT:\nNONE\nIndicate by check mark: YES NO\n• if the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act. þ ¨\n• if the registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Act. ¨ þ\n• whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities \nExchange Act of 1934 during the preceding 12 months (or for such shorter period that the registrant was required \nto file such reports), and (2) has been subject to such filing requirements for the past 90 days.\nþ ¨\n• whether the registrant has submitted electronically every Interactive Data File required to be submitted pursuan

# Embeddings -MPNET


In [20]:
# pip install -qU langchain-mistralai

In [61]:
# !pip install sentence-transformers

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [25]:
# vector_1 = embeddings.embed_query(all_splits[0].page_content)
# vector_2 = embeddings.embed_query(all_splits[1].page_content)

# assert len(vector_1) == len(vector_2)
# print(f"Generated vectors of length {len(vector_1)}\n")
# print(vector_1[:10])

### Vector stores

In [29]:
# pip install -qU langchain-chroma

In [27]:
from langchain_chroma import Chroma

vector_store = Chroma(embedding_function=embeddings,
                     persist_directory='v3')


In [28]:
ids = vector_store.add_documents(documents=all_splits)

In [39]:
# ids

### Usage

In [32]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

results[0]

Document(id='82ab4b66-2448-4924-969a-3d6dedf80fda', metadata={'page': 27, 'page_label': '28', 'source': 'Nike.pdf', 'start_index': 877}, page_content='our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising \nstrategies in the region, among other functions.\nIn the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of \nwhich are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one \nlocated in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is \nlocated in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of \nwhich are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United \nStates are located in Laakda

In [38]:
results = vector_store.similarity_search(
    "what was the translational exposures"
)

results[0]

Document(id='0a3c79db-1aae-4780-9700-d9dbd3d46156', metadata={'page': 47, 'page_label': '48', 'source': 'Nike.pdf', 'start_index': 0}, page_content="Certain currency forward contracts used to manage the foreign exchange exposure of non-functional currency denominated \nmonetary assets and liabilities subject to remeasurement are not formally designated as hedging instruments. Accordingly, \nchanges in fair value of these instruments are recognized in Other (income) expense, net and are intended to offset the foreign \ncurrency impact of the remeasurement of the related non-functional currency denominated asset or liability being hedged. \nTRANSLATIONAL EXPOSURES\nMany of our foreign subsidiaries operate in functional currencies other than the U.S. Dollar. Fluctuations in currency exchange \nrates create volatility in our reported results as we are required to translate the balance sheets, operational results and cash flows \nof these subsidiaries into U.S. Dollars for consolidated repo

In [40]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How many distribution centers does Nike have in the US?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.44210007786750793

page_content='our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 
States are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we own.' metadata={'page': 27, 'page_lab

# Generation

In [42]:
doc=results[0][0]

In [44]:
doc.page_content

'our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising \nstrategies in the region, among other functions.\nIn the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of \nwhich are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one \nlocated in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is \nlocated in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of \nwhich are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United \nStates are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we own.'

In [48]:
# Generate response using Ollama
# Set up the generation prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Based on the following context, answer the question as accurately as possible:\n\n"
        "give complete information provided in the context"
        "Always say have a gooday at the end"
        "Context: {context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# Use the retrieved document's page content for the context
context = doc.page_content  # Corrected to use doc (from similarity search)
question = "How many distribution centers does Nike have in the US?"
prompt = prompt_template.format(context=context, question=question)


print('-------------')
print(prompt)
print('-----------------------')

# Generate answer with Ollama
response = ollama.generate(
    model="llama3.2",
    prompt=prompt
    
)

# Print the generated answer
print(f"Generated Answer: {response.response}")

-------------
Based on the following context, answer the question as accurately as possible:

give complete information provided in the contextAlways say have a gooday at the endContext: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 

In [50]:
# Generate response using Ollama
# Set up the generation prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Based on the following context, answer the question as accurately as possible:\n\n"
        "give complete information provided in the context"
        "Always say have a gooday at the end"
        "Context: {context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# Use the retrieved document's page content for the context
context = doc.page_content  # Corrected to use doc (from similarity search)
question = "when was nike incorporated?"
prompt = prompt_template.format(context=context, question=question)

# Generate answer with Ollama
response = ollama.generate(
    model="llama3.2",
    prompt=prompt
    
)

# Print the generated answer
print(f"Generated Answer: {response.response}")

Generated Answer: The context doesn't explicitly mention the incorporation date of NIKE. However, it does provide information about their distribution centers and facilities outside the United States, which gives an idea of their global presence but not the exact date of incorporation.


- Note it can not produce answert of the above question since context or is fixed with retreival .only distribution center
  context is there although the whole document has the full information



### 1. **Retriever Step**: 
The retriever searches a large corpus (e.g., a document database, knowledge base, or vector store) for documents or chunks of text that are most relevant to the user’s query. This step is typically based on similarity search (e.g., cosine similarity between the query embedding and the embeddings of the stored documents).

### 2. **Context Formation**: 
The documents or passages retrieved by the retriever are then used to **form the context** for the generative model. The retrieved information is passed to the **generator** (typically a language model, such as GPT or BERT) along with the user's query. This allows the generator to use the relevant context to craft a more accurate and informed response.

### 3. **Generation Step**: 
The generative model uses both the query and the retrieved context to generate a response. The context provides factual information that the model can use to produce a relevant, accurate, and coherent answer. This step is what distinguishes RAG from traditional language models, which generate responses solely based on the input query without relying on external knowledge.

### Example Flow:
1. **Input Query**: "What is the capital of France?"
2. **Retriever**: The retriever searches the knowledge base and retrieves relevant documents, such as:
   - "France is a country in Europe, and its capital is Paris."
3. **Context for Generation**: The retrieved context ("France's capital is Paris") is combined with the input query.
4. **Generator**: The generative model uses this combined context to generate the final answer: "The capital of France is Paris."

In essence, **the retriever forms the context by providing relevant information**, and **the generator uses this context to produce a response**. This architecture allows RAG to generate more accurate, relevant, and contextually grounded answers compared to a purely generative model that might not have access to external information.

### Complete retriver to generation 

In [71]:
query = input()
result = vector_store.similarity_search(query,k=1)
doc=result[0]
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Based on the following context, answer the question"
        
        "Context: {context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# Use the retrieved document's page content for the context
context = doc.page_content  # Corrected to use doc (from similarity search)

prompt = prompt_template.format(context=context, question=query)

# Generate answer with Ollama
response = ollama.generate(
    model="llama3.2",
    prompt=prompt
    
)

# Print the generated answer
print(f"Generated Answer: {response.response}")

 term SOFR


Generated Answer: The question is asking about the term SOFR.

Unfortunately, the provided context does not mention the term SOFR (Sovereign Credit Risk), which is likely a typo or omission in the text. However, based on common finance and accounting terms, it appears that SOFR might be intended to refer to "Sofr" or more broadly, "Sovereign Credit Risk", but there's no clear indication of this.

However, according to recent accounting standards issued by the Financial Accounting Standards Board (FASB), SOFR is indeed a relevant term. In 2022, FASB issued Accounting Standards Update ASU 2022-04, which includes guidance related to financial instruments affected by changes in sovereign credit risk, such as SOFR.

Considering this context, the answer could be: "SOFR"


# Adding memory

In [63]:
query = input()
result = vector_store.similarity_search(query,k=1)
doc=result[0]
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Based on the following context, answer the question"
        
        "Context: {context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# Use the retrieved document's page content for the context
context = doc.page_content  # Corrected to use doc (from similarity search)

prompt = prompt_template.format(context=context, question=query)

# Generate answer with Ollama
llm= ollama.generate(
    model="llama3.2",
    prompt=prompt
    
)

#memory
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


from langchain.chains import ConversationalRetrievalChain
qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vector_store.as_retriever(),
    memory=memory
)


result = qa({"question": query})


# Print the generated answer
print(f"Generated Answer: result['answer']")

 when was nike incorporated?


ValidationError: 2 validation errors for LLMChain
llm.is-instance[Runnable]
  Input should be an instance of Runnable [type=is_instance_of, input_value=GenerateResponse(model='l... 11, 220, 4468, 16, 13]), input_type=GenerateResponse]
    For further information visit https://errors.pydantic.dev/2.10/v/is_instance_of
llm.is-instance[Runnable]
  Input should be an instance of Runnable [type=is_instance_of, input_value=GenerateResponse(model='l... 11, 220, 4468, 16, 13]), input_type=GenerateResponse]
    For further information visit https://errors.pydantic.dev/2.10/v/is_instance_of